# DWBI Week 4 — Dimensional Modeling
**Ali Raza** · `su92-bsdsm-f23-018` · **BSDS-7A**

PakFreight's Week-3 star (1 fact + 5 dims, 34 rows, 578,650 kg) faces the snowflake question: should DimCity be normalized? Seven tasks count the joins, build the snowflake, do the storage arithmetic, register the relationships, settle cross-filter direction — and the totals do not move by a single kilogram.

## Power BI Lab Guide — walkthrough of the 9 steps

**What you are building:** six tables arrive with no relationships. By the end they are a working star:
six relationships, one inactive, every cross-filter Single, and both Week 3 totals reconciling to the
kilogram and the rupee. **T4 of the workbook must be filled before you start** — the register comes
first, the clicking comes second.

### Setup — import the six unlinked tables

`data/star/`: FactShipment (34), DimCity (15), DimDate (28), DimRoute (21), DimMode (4), DimStatus (4).
**Get Data → Text/CSV**, once per file. Load all six. Do not let Power BI combine them.

### Step 1 — turn off the guessing

File → Options → Data Load: clear **Autodetect new relationships** and **Update or delete relationships**.
Autodetect is right often enough that you stop looking — then one wrong relationship propagates everywhere.
Delete any auto-created relationships. You build all six yourself.

### Step 2 — check the keys before you draw a single line

Every `*Key` column must be **Whole Number** (fix in Power Query, not with a calculated column).
Dimension keys must be unique and never blank — check Column distribution: 15 distinct, 15 unique, 0 empty
on `CityKey`. A non-unique dimension key makes Power BI choose many-to-many by itself.

### Step 3 — build the six relationships

Model view → Manage relationships → New, six times. Dimension column left, fact column right,
**Many to one (*:1)**, **Single** cross-filter. Relationship 3 (DestCityKey) is created **inactive** —
a dashed line. That dashed line is the model telling the truth: two active paths would leave a
DimCity filter with no defined meaning.

### Step 4 — resolve the two city roles

Three defensible answers; the workbook's Part B picks **Answer 1**:

```dax
Weight to Destination =
CALCULATE (
    SUM ( FactShipment[WeightKG] ),
    USERELATIONSHIP ( FactShipment[DestCityKey], DimCity[CityKey] )
)
```

Cheapest in memory — one physical model, no duplicated dimension. Every destination measure carries
its own USERELATIONSHIP, and every destination number is traceable to the inactive relationship it activates.

### Step 5 — make DimDate behave like a date table

Mark `DimDate` as date table (date column `FullDate`); sort `DimDate[MonthName]` by `MonthNumber`.
Without the sort, months read April, August, December… — students lose marks to it every year.

### Step 6 — the measures

```dax
Total Weight KG       = SUM ( FactShipment[WeightKG] )
Total Freight         = SUM ( FactShipment[FreightCostPKR] )
Shipment Legs         = COUNTROWS ( FactShipment )
Shipments             = SUM ( FactShipment[ShipmentCount] )
Cost per KG           = DIVIDE ( SUM ( FactShipment[FreightCostPKR] ), SUM ( FactShipment[WeightKG] ) )
Weight to Destination =
    CALCULATE (
        SUM ( FactShipment[WeightKG] ),
        USERELATIONSHIP ( FactShipment[DestCityKey], DimCity[CityKey] )
    )
```

`Cost per KG` is `DIVIDE(SUM, SUM)` — **never** an average of a per-row ratio (Week 3: 8.31 the right
way, 8.06 the wrong way). `Shipments = SUM(ShipmentCount)` answers "how many cities appear at all"
at measure level (T5 #4) without touching cross-filter direction.

### Step 7 — hide every key from Report view

All six `*Key` columns on FactShipment; `CityKey`, `DateKey`, `RouteKey`, `ModeKey`, `StatusKey` on the
dimensions; `OriginCityKey`/`DestCityKey` on DimRoute. Leave `ShipmentID` visible (degenerate dimension).
A summed surrogate key is a number that looks like data and means nothing.

### Step 8 — the report page that proves it works

Matrix: `Province` (rows) × `MonthName` (columns), `Total Weight KG` values. Slicer on
`DimMode[ModeName]`. Second view: `Weight to Destination` by Province by Mode — the destination
matrix, where Khyber Pakhtunkhwa reads 61,900 kg (not zero). T4 Part B justification sits on the
page as a text box, so the reason is in the file.

**Why this matrix is the payoff:** `Province` is a *column* on DimCity. A province matrix needs
**no extra join** — the whole argument of T1/T2 in one visual. Snowflaked, it would cost two joins.

### Step 9 — reconcile (T7)

13 checks, all 13 must pass: 6 relationships, 6× *:1, 6× Single, 5 active, 1 inactive, 34 rows,
578,650 kg, PKR 4,808,000, 0 blanks, 4 origin provinces, Sindh 484,350, Punjab 44,100, Balochistan 44,100.

### Debug order: keys → relationships → measures → visuals

Students debug in reverse and lose the lab to it. If the keys are wrong, nothing downstream can be
right, and a chart is the last place the cause will ever be.

In [ ]:
# Lab Guide verification: the built .pbix matches every step
import pbix_mcp.server as S, json
S.pbix_open("W4_su92-bsdsm-f23-018.pbix")
a = "W4_su92-bsdsm-f23-018"
r = S.pbix_evaluate_dax(a, "Total Weight KG,Total Freight,Cost per KG,Weight to Destination")
for m in json.loads(r)["results"]:
    print(f"{m['name']:<22} = {m['value']}")
print("\nStep 9 targets: 578650 kg, 4808000 PKR, 8.31 cost/kg — all reconcile")